In [1]:
!pip install tqdm

import nltk as nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from tqdm import tqdm
from collections import defaultdict, Counter
import numpy as np
import math as math

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

stop_words = set(stopwords.words('english'))
df = pd.DataFrame(pd.read_json('/content/drive/MyDrive/Information_Retrieval/BM25/data/corpus.jsonl', lines=True))
df.drop(columns=['metadata'], inplace=True)
corpus_tokens = {}

def tokenize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return filtered_tokens

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    tokens = tokenize(row['text'])
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    corpus_tokens[row['_id']] = filtered_tokens

100%|██████████| 171332/171332 [03:32<00:00, 805.14it/s]


In [4]:
inverted_index = defaultdict(dict)
for doc_id, tokens in tqdm(corpus_tokens.items(), desc='Indexing...'):
    for term, frequency in Counter(tokens).items():
        inverted_index[term][doc_id] = frequency

Indexing...: 100%|██████████| 171332/171332 [00:15<00:00, 11214.95it/s]


In [5]:
docs_len = {}
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc='Calculating doc stats...'):
    docs_len[row['_id']] = len(corpus_tokens[row['_id']])

Calculating doc stats...: 100%|██████████| 171332/171332 [00:07<00:00, 21526.58it/s]


In [6]:
N = len(df)
average_dl = sum(docs_len.values()) / N

def bm25_score(term, doc_id, k1=0.25, b=0.75):
  if term not in inverted_index or doc_id not in inverted_index[term]:
    return 0.0

  tf = inverted_index[term][doc_id]
  dl = docs_len[doc_id]
  df = len(inverted_index[term])
  idf = math.log((N - df + 0.5) / (df + 0.5))
  denom = tf + k1 * (1 - b + b * dl / average_dl)
  score = idf * (tf * (k1 + 1) / denom)
  return score


In [7]:
query = 'what is the origin of COVID-19'
query_tokens = tokenize(query)
union_docs = set().union(*(inverted_index[t].keys() for t in query_tokens))

scores = defaultdict(float)
for doc_id in tqdm(union_docs, desc='Calculating scores...'):
    score = sum(bm25_score(t, doc_id) for t in query_tokens)
    scores[doc_id] = score

sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
sorted_scores = sorted_scores[:50]
sorted_scores

Calculating scores...: 100%|██████████| 2048/2048 [00:00<00:00, 253039.58it/s]


[('dv9m19yk', 5.337036940728824),
 ('vh96sjss', 5.3160074738276),
 ('e3wjo0yk', 5.192911569094133),
 ('8ccl9aui', 5.188475778797673),
 ('4uaa6kpg', 5.179876303355314),
 ('icwvm7jp', 5.15709701599889),
 ('deajwhx0', 5.15709701599889),
 ('d0x23frk', 5.1445282166581086),
 ('ymhcouo5', 5.134517202131886),
 ('021q9884', 5.13287905209271),
 ('2vpvdm11', 5.13287905209271),
 ('wim5q9a5', 5.118027022093672),
 ('49360l2a', 5.117091360708405),
 ('v6ci69n0', 5.115532685077529),
 ('73ylxhb7', 5.112228892110243),
 ('l0kc731z', 5.092683234168169),
 ('9l97eihy', 5.092401494230845),
 ('cniyembt', 5.082872786421372),
 ('zd7smm8r', 5.077981722903272),
 ('ayj4z8qn', 5.068227780579463),
 ('us1spoxu', 5.0655163742448375),
 ('z14rf85c', 5.050257805671744),
 ('hewbl5yu', 5.04883187974919),
 ('fyrrwy9v', 5.04883187974919),
 ('jkhvcjcb', 5.04883187974919),
 ('9t0bafyz', 5.039189493503699),
 ('7csfkoh8', 5.034932841632498),
 ('42wv7zl6', 5.030212897720992),
 ('2ntxpdke', 5.027803400958268),
 ('0pbp97ik', 5.02001